In [47]:
import boto3
import json
import time
import zipfile
from io import BytesIO
import uuid
import pprint
import logging

print(boto3.__version__)

1.43.46


In [41]:
logging.basicConfig(
    format="[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

In [42]:
sts_client = boto3.client("sts", region_name="us-east-1")
iam_client = boto3.client("iam", region_name="us-east-1")
lambda_client = boto3.client("lambda", region_name="us-east-1")

# AgentCore replaces the maintenance-mode-blocked bedrock-agent clients.
# Control plane (create/get/delete harness) vs data plane (invoke_harness).
agentcore_control_client = boto3.client(
    "bedrock-agentcore-control", region_name="us-east-1"
)
agentcore_client = boto3.client("bedrock-agentcore", region_name="us-east-1")

In [43]:
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()["Account"]
region, account_id

('us-east-1', '623271127785')

In [44]:
# inference_profile = "us.amazon.nova-micro-v1:0"
inference_profile = "us.anthropic.claude-sonnet-4-6"
foundation_model = inference_profile[3:]
foundation_model

'anthropic.claude-sonnet-4-6'

In [45]:
suffix = f"{region}-{account_id}"

# Kept for the Lambda naming below (Lambda role/function still use these).
agent_name = "hr-assistant-function-def"
agent_instruction = "You are an HR agent, helping employees understand HR policies and manage vacation time"
lambda_function_role = f"{agent_name}-lambda-role-{suffix}"
lambda_function_name = f"{agent_name}-{suffix}"

# AgentCore Harness naming. Harness name must match ^[a-zA-Z][a-zA-Z0-9_]{0,47}$
# so we replace hyphens with underscores.
harness_name = "hr_assistant_harness"
harness_execution_role_name = f"AgentCoreHarnessRole_{harness_name}"

In [9]:
# creating employee database to be used by lambda function
import sqlite3
import random
from datetime import date, timedelta

# Connect to the SQLite database (creates a new one if it doesn't exist)
conn = sqlite3.connect("employee_database.db")
c = conn.cursor()

# Create the employees table
c.execute(
    """CREATE TABLE IF NOT EXISTS employees
                (employee_id INTEGER PRIMARY KEY AUTOINCREMENT, employee_name TEXT, employee_job_title TEXT, employee_start_date TEXT, employee_employment_status TEXT)"""
)

# Create the vacations table
c.execute(
    """CREATE TABLE IF NOT EXISTS vacations
                (employee_id INTEGER, year INTEGER, employee_total_vacation_days INTEGER, employee_vacation_days_taken INTEGER, employee_vacation_days_available INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))"""
)

# Create the planned_vacations table
c.execute(
    """CREATE TABLE IF NOT EXISTS planned_vacations
                (employee_id INTEGER, vacation_start_date TEXT, vacation_end_date TEXT, vacation_days_taken INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))"""
)

# Generate some random data for 10 employees
employee_names = [
    "John Doe",
    "Jane Smith",
    "Bob Johnson",
    "Alice Williams",
    "Tom Brown",
    "Emily Davis",
    "Michael Wilson",
    "Sarah Taylor",
    "David Anderson",
    "Jessica Thompson",
]
job_titles = [
    "Manager",
    "Developer",
    "Designer",
    "Analyst",
    "Accountant",
    "Sales Representative",
]
employment_statuses = ["Active", "Inactive"]

for i in range(10):
    name = employee_names[i]
    job_title = random.choice(job_titles)
    start_date = date(
        2015 + random.randint(0, 7), random.randint(1, 12), random.randint(1, 28)
    ).strftime("%Y-%m-%d")
    employment_status = random.choice(employment_statuses)
    c.execute(
        "INSERT INTO employees (employee_name, employee_job_title, employee_start_date, employee_employment_status) VALUES (?, ?, ?, ?)",
        (name, job_title, start_date, employment_status),
    )
    employee_id = c.lastrowid

    # Generate vacation data for the current employee
    for year in range(date.today().year, date.today().year - 3, -1):
        total_vacation_days = random.randint(10, 30)
        days_taken = random.randint(0, total_vacation_days)
        days_available = total_vacation_days - days_taken
        c.execute(
            "INSERT INTO vacations (employee_id, year, employee_total_vacation_days, employee_vacation_days_taken, employee_vacation_days_available) VALUES (?, ?, ?, ?, ?)",
            (employee_id, year, total_vacation_days, days_taken, days_available),
        )

        # Generate some planned vacations for the current employee and year
        num_planned_vacations = random.randint(0, 3)
        for _ in range(num_planned_vacations):
            start_date = date(
                year, random.randint(1, 12), random.randint(1, 28)
            ).strftime("%Y-%m-%d")
            end_date = (
                date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:]))
                + timedelta(days=random.randint(1, 14))
            ).strftime("%Y-%m-%d")
            days_taken = date(
                int(end_date[:4]), int(end_date[5:7]), int(end_date[8:])
            ) - date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:]))
            c.execute(
                "INSERT INTO planned_vacations (employee_id, vacation_start_date, vacation_end_date, vacation_days_taken) VALUES (?, ?, ?, ?)",
                (employee_id, start_date, end_date, days_taken.days),
            )

# Commit the changes and close the connection
conn.commit()
conn.close()

In [23]:
%%writefile lambda_function.py
import json
import os
import shutil
import sqlite3

# /var/task is read-only in Lambda, so copy the bundled DB to /tmp on cold
# start and mutate that copy. Writes therefore persist only for the lifetime
# of a single Lambda container -- fine for this demo, not for production.
BUNDLED_DB = "/var/task/employee_database.db"
DB_PATH = "/tmp/employee_database.db"

if not os.path.exists(DB_PATH):
    shutil.copyfile(BUNDLED_DB, DB_PATH)


def _connect():
    return sqlite3.connect(DB_PATH)


def get_available_vacations_days(employee_id):
    with _connect() as conn:
        row = conn.execute(
            """
            SELECT employee_vacation_days_available
            FROM vacations
            WHERE employee_id = ?
            ORDER BY year DESC
            LIMIT 1
            """,
            (employee_id,),
        ).fetchone()
    if row is None:
        return {"error": f"No vacation data found for employee_id {employee_id}"}
    return {"employee_id": employee_id, "available_vacation_days": row[0]}


def reserve_vacation_time(employee_id, start_date, end_date=None):
    """Reserve vacation time for an employee.

    - Adds a row to planned_vacations covering [start_date, end_date] (inclusive
      of start_date, exclusive of end_date to match the seed data's convention).
    - Decrements employee_vacation_days_available on the current-year row of
      vacations by the number of days reserved.
    - If end_date is omitted, reserves a single day (start_date only).
    """
    from datetime import date as _date

    if end_date is None:
        end_date = start_date
    try:
        start = _date.fromisoformat(start_date)
        end = _date.fromisoformat(end_date)
    except ValueError as exc:
        return {"error": f"Invalid date: {exc}. Use YYYY-MM-DD."}

    days = (end - start).days + 1  # inclusive of both endpoints
    if days <= 0:
        return {"error": "end_date must be on or after start_date"}

    with _connect() as conn:
        row = conn.execute(
            """
            SELECT employee_vacation_days_available, year
            FROM vacations
            WHERE employee_id = ?
            ORDER BY year DESC
            LIMIT 1
            """,
            (employee_id,),
        ).fetchone()
        if row is None:
            return {"error": f"No vacation record for employee_id {employee_id}"}
        available, year = row
        if available < days:
            return {
                "error": (
                    f"Not enough vacation days: requested {days}, "
                    f"available {available}"
                )
            }

        conn.execute(
            """
            INSERT INTO planned_vacations
                (employee_id, vacation_start_date, vacation_end_date, vacation_days_taken)
            VALUES (?, ?, ?, ?)
            """,
            (employee_id, start_date, end_date, days),
        )
        conn.execute(
            """
            UPDATE vacations
            SET employee_vacation_days_taken =
                    employee_vacation_days_taken + ?,
                employee_vacation_days_available =
                    employee_vacation_days_available - ?
            WHERE employee_id = ? AND year = ?
            """,
            (days, days, employee_id, year),
        )
        conn.commit()

    return {
        "employee_id": employee_id,
        "reserved_start_date": start_date,
        "reserved_end_date": end_date,
        "days_reserved": days,
        "remaining_vacation_days": available - days,
        "status": "confirmed",
    }


# Dispatch table: the notebook passes tool_name in the event so one Lambda
# can serve multiple AgentCore tools.
TOOLS = {
    "get_available_vacations_days": get_available_vacations_days,
    "reserve_vacation_time": reserve_vacation_time,
}


def lambda_handler(event, context):
    tool_name = event.get("tool_name")
    if not tool_name:
        # Back-compat: older callers sent {"employee_id": N} with no tool_name.
        tool_name = "get_available_vacations_days"

    fn = TOOLS.get(tool_name)
    if fn is None:
        return {"statusCode": 400, "body": {"error": f"unknown tool: {tool_name}"}}

    kwargs = {k: v for k, v in event.items() if k != "tool_name"}
    try:
        if "employee_id" in kwargs:
            kwargs["employee_id"] = int(kwargs["employee_id"])
    except (TypeError, ValueError):
        return {"statusCode": 400, "body": {"error": "employee_id must be an integer"}}

    try:
        result = fn(**kwargs)
    except TypeError as exc:
        return {"statusCode": 400, "body": {"error": f"bad arguments: {exc}"}}
    return {"statusCode": 200, "body": result}

Overwriting lambda_function.py


In [24]:
try:
    assume_role_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "lambda.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }

    assume_role_policy_document_json = json.dumps(assume_role_policy_document)

    lambda_iam_role = iam_client.create_role(
        RoleName=lambda_function_role,
        AssumeRolePolicyDocument=assume_role_policy_document_json,
    )

    # Pause to make sure role is created
    time.sleep(10)
except:
    lambda_iam_role = iam_client.get_role(RoleName=lambda_function_role)

iam_client.attach_role_policy(
    RoleName=lambda_function_role,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)

{'ResponseMetadata': {'RequestId': 'b1f6d7ce-7128-4b4c-8993-105ddf716fe9',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 13 Aug 2026 04:29:28 GMT',
   'x-amzn-requestid': 'b1f6d7ce-7128-4b4c-8993-105ddf716fe9',
   'content-type': 'text/xml',
   'content-length': '212'},
  'RetryAttempts': 0}}

In [25]:
# Package up the lambda function code
s = BytesIO()
z = zipfile.ZipFile(s, "w")
z.write("lambda_function.py")
z.write("employee_database.db")
z.close()
zip_content = s.getvalue()

# Create or update -- the function already exists from an earlier run in this
# account, so update_function_code is the normal path. Fall back to
# create_function for a fresh account.
try:
    lambda_function = lambda_client.update_function_code(
        FunctionName=lambda_function_name,
        ZipFile=zip_content,
    )
    print("Updated Lambda:", lambda_function["FunctionArn"])
except lambda_client.exceptions.ResourceNotFoundException:
    lambda_function = lambda_client.create_function(
        FunctionName=lambda_function_name,
        Runtime="python3.14",
        Timeout=180,
        Role=lambda_iam_role["Role"]["Arn"],
        Code={"ZipFile": zip_content},
        Handler="lambda_function.lambda_handler",
    )
    print("Created Lambda:", lambda_function["FunctionArn"])

Updated Lambda: arn:aws:lambda:us-east-1:623271127785:function:hr-assistant-function-def-us-east-1-623271127785


In [26]:
# --- Create the AgentCore Harness execution role ---
# The harness (microVM) assumes this role. It needs Bedrock model invocation
# permissions plus the CloudWatch/X-Ray/ECR-public/workload-identity bits from
# the sample execution-role policy in the AgentCore docs.

harness_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

harness_permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockModelInvocation",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream",
            ],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                f"arn:aws:bedrock:{region}:{account_id}:*",
            ],
        },
        {
            "Sid": "EcrPublicTokenAccess",
            "Effect": "Allow",
            "Action": ["ecr-public:GetAuthorizationToken", "sts:GetServiceBearerToken"],
            "Resource": "*",
        },
        {
            "Sid": "XRayTracingAccess",
            "Effect": "Allow",
            "Action": [
                "xray:PutTraceSegments",
                "xray:PutTelemetryRecords",
                "xray:GetSamplingRules",
                "xray:GetSamplingTargets",
            ],
            "Resource": "*",
        },
        {
            "Sid": "CloudWatchLogs",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
                "logs:DescribeLogStreams",
                "logs:DescribeLogGroups",
                "logs:PutResourcePolicy",
            ],
            "Resource": "*",
        },
        {
            "Sid": "CloudWatchMetricsPublish",
            "Effect": "Allow",
            "Action": "cloudwatch:PutMetricData",
            "Resource": "*",
            "Condition": {
                "StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}
            },
        },
        {
            "Sid": "AgentCoreWorkloadIdentity",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetWorkloadAccessToken",
                "bedrock-agentcore:GetWorkloadAccessTokenForJWT",
            ],
            # AgentCore names workload identities after the harness (e.g.
            # `hr_assistant_harness-<suffix>`), not `harness_<name>-*`.
            "Resource": [
                f"arn:aws:bedrock-agentcore:{region}:{account_id}:workload-identity-directory/default",
                f"arn:aws:bedrock-agentcore:{region}:{account_id}:workload-identity-directory/default/workload-identity/*",
            ],
        },
        {
            "Sid": "AgentCoreMemoryDefault",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateEvent",
                "bedrock-agentcore:DeleteEvent",
                "bedrock-agentcore:GetEvent",
                "bedrock-agentcore:ListEvents",
                "bedrock-agentcore:ListActors",
                "bedrock-agentcore:ListSessions",
                "bedrock-agentcore:RetrieveMemoryRecords",
                "bedrock-agentcore:ListMemoryRecords",
                "bedrock-agentcore:GetMemoryRecord",
            ],
            # The harness's default memory is named `{harnessName}-<suffix>`
            # (e.g. `hr_assistant_harness-13Bg7DBdLs`), not `harness_<name>-*`.
            # Also allow sub-resources (actor/session/event/memoryRecord).
            "Resource": [
                f"arn:aws:bedrock-agentcore:{region}:{account_id}:memory/{harness_name}-*",
                f"arn:aws:bedrock-agentcore:{region}:{account_id}:memory/{harness_name}-*/*",
            ],
        },
    ],
}

try:
    harness_role = iam_client.create_role(
        RoleName=harness_execution_role_name,
        AssumeRolePolicyDocument=json.dumps(harness_trust_policy),
    )
    time.sleep(10)  # let the role propagate
except iam_client.exceptions.EntityAlreadyExistsException:
    harness_role = iam_client.get_role(RoleName=harness_execution_role_name)

# put_role_policy overwrites the inline policy in place, so re-running this
# cell after tweaking permissions is the intended way to update the live role.
iam_client.put_role_policy(
    RoleName=harness_execution_role_name,
    PolicyName=f"{harness_execution_role_name}-inline",
    PolicyDocument=json.dumps(harness_permissions_policy),
)

harness_execution_role_arn = harness_role["Role"]["Arn"]
print("Harness execution role:", harness_execution_role_arn)

Harness execution role: arn:aws:iam::623271127785:role/AgentCoreHarnessRole_hr_assistant_harness


In [27]:
# --- Create (or update) the harness ---
# Inline function tools: the harness returns control to *this notebook* when
# the model wants to call one of these, and we invoke the Lambda.
# (The other option would be AgentCore Gateway wrapping the Lambda as an MCP
# tool; then the harness executes end-to-end without client involvement.)

vacation_tools = [
    {
        "type": "inline_function",
        "name": "get_available_vacations_days",
        "config": {
            "inlineFunction": {
                "description": "Get the number of remaining vacation days for an employee this year.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "employee_id": {
                            "type": "integer",
                            "description": "The numeric employee ID.",
                        }
                    },
                    "required": ["employee_id"],
                },
            }
        },
    },
    {
        "type": "inline_function",
        "name": "reserve_vacation_time",
        "config": {
            "inlineFunction": {
                "description": (
                    "Reserve vacation time for an employee. Pass start_date, "
                    "and optionally end_date, as ISO dates (YYYY-MM-DD). "
                    "Omit end_date to reserve a single day. Decrements the "
                    "employee's remaining balance and records a planned vacation."
                ),
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "employee_id": {
                            "type": "integer",
                            "description": "The numeric employee ID.",
                        },
                        "start_date": {
                            "type": "string",
                            "description": "First day of the vacation, YYYY-MM-DD.",
                        },
                        "end_date": {
                            "type": "string",
                            "description": (
                                "Last day of the vacation (inclusive), YYYY-MM-DD. "
                                "Omit for a single-day reservation."
                            ),
                        },
                    },
                    "required": ["employee_id", "start_date"],
                },
            }
        },
    },
]

try:
    create_resp = agentcore_control_client.create_harness(
        harnessName=harness_name,
        executionRoleArn=harness_execution_role_arn,
        model={"bedrockModelConfig": {"modelId": inference_profile}},
        systemPrompt=[{"text": agent_instruction}],
        tools=vacation_tools,
    )
    harness_arn = create_resp["harness"]["arn"]
    harness_id = create_resp["harness"]["harnessId"]
except agentcore_control_client.exceptions.ConflictException:
    # Already created -- look it up, then update in place so it picks up any
    # tool/model/prompt changes on re-run.
    summary = next(
        h
        for h in agentcore_control_client.list_harnesses()["harnesses"]
        if h["harnessName"] == harness_name
    )
    harness_arn = summary["arn"]
    harness_id = summary["harnessId"]
    agentcore_control_client.update_harness(
        harnessId=harness_id,
        executionRoleArn=harness_execution_role_arn,
        model={"bedrockModelConfig": {"modelId": inference_profile}},
        systemPrompt=[{"text": agent_instruction}],
        tools=vacation_tools,
    )
    print("Updated existing harness with latest tool list.")

print("Harness ARN:", harness_arn)

Updated existing harness with latest tool list.
Harness ARN: arn:aws:bedrock-agentcore:us-east-1:623271127785:harness/hr_assistant_harness-I5IxgEpmqy


In [28]:
# --- Wait for the harness to become READY before we invoke it ---
harness_id = harness_arn.split("/")[-1]
for _ in range(60):
    status = agentcore_control_client.get_harness(harnessId=harness_id)["harness"][
        "status"
    ]
    print("harness status:", status)

    if status == "READY":
        break
    if status in ("CREATE_FAILED", "FAILED"):
        raise RuntimeError(f"Harness creation failed: {status}")
    time.sleep(5)
else:
    raise TimeoutError("Harness did not become READY within 5 minutes")

harness status: UPDATING
harness status: UPDATING
harness status: READY


In [29]:
# --- Invoke the harness ---
# 1. Ask a question that needs the tool.
# 2. Read the stream. If stopReason is "tool_use", capture the toolUse block.
# 3. Invoke the Lambda with the tool name + input.
# 4. Send a second invoke_harness call with the assistant's toolUse + our toolResult.
# 5. Stream the final answer.


def invoke_lambda_tool(tool_name, tool_input):
    """Call the shared Lambda, dispatching on tool_name."""
    payload = {"tool_name": tool_name, **tool_input}
    resp = lambda_client.invoke(
        FunctionName=lambda_function_name,
        InvocationType="RequestResponse",
        Payload=json.dumps(payload).encode(),
    )
    payload = json.loads(resp["Payload"].read())
    body = payload.get("body", payload)
    return body if isinstance(body, (dict, list)) else json.loads(body)


def collect_stream(stream):
    """Return (assistant_content_blocks, stop_reason, text_out)."""
    content_blocks = []
    current_text = None
    current_tool = None  # {"toolUseId", "name", "input_json"}
    stop_reason = None
    printed_text = []
    for event in stream:
        if "contentBlockStart" in event:
            start = event["contentBlockStart"].get("start", {})
            if "toolUse" in start:
                current_tool = {
                    "toolUseId": start["toolUse"]["toolUseId"],
                    "name": start["toolUse"]["name"],
                    "input_json": "",
                }
        elif "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                if current_text is None:
                    current_text = ""
                current_text += delta["text"]
                print(delta["text"], end="", flush=True)
                printed_text.append(delta["text"])
            elif "toolUse" in delta and current_tool is not None:
                current_tool["input_json"] += delta["toolUse"].get("input", "")
        elif "contentBlockStop" in event:
            if current_text is not None:
                content_blocks.append({"text": current_text})
                current_text = None
            if current_tool is not None:
                content_blocks.append(
                    {
                        "toolUse": {
                            "toolUseId": current_tool["toolUseId"],
                            "name": current_tool["name"],
                            "input": json.loads(current_tool["input_json"] or "{}"),
                        }
                    }
                )
                current_tool = None
        elif "messageStop" in event:
            stop_reason = event["messageStop"].get("stopReason")
        elif "runtimeClientError" in event:
            raise RuntimeError(event["runtimeClientError"])
    return content_blocks, stop_reason, "".join(printed_text)


session_id = str(uuid.uuid4()) + "-hr-session"  # min 33 chars

messages = [
    {
        "role": "user",
        "content": [
            {"text": "How many vacation days does employee 1 have left this year?"}
        ],
    }
]

print("=== Assistant ===")
resp = agentcore_client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=messages,
)
assistant_blocks, stop_reason, _ = collect_stream(resp["stream"])

# Handle tool round-trips (usually just one, but loop in case the model chains).
while stop_reason == "tool_use":
    tool_use = next(b["toolUse"] for b in assistant_blocks if "toolUse" in b)
    print(f"\n\n>>> Calling Lambda tool={tool_use['name']} with {tool_use['input']}")
    tool_result = invoke_lambda_tool(tool_use["name"], tool_use["input"])
    print(f">>> Lambda returned: {tool_result}\n")

    followup_messages = [
        {"role": "assistant", "content": assistant_blocks},
        {
            "role": "user",
            "content": [
                {
                    "toolResult": {
                        "toolUseId": tool_use["toolUseId"],
                        "content": [{"text": json.dumps(tool_result)}],
                        "status": "success",
                    }
                }
            ],
        },
    ]

    print("=== Assistant (after tool result) ===")
    resp = agentcore_client.invoke_harness(
        harnessArn=harness_arn,
        runtimeSessionId=session_id,
        messages=followup_messages,
    )
    assistant_blocks, stop_reason, _ = collect_stream(resp["stream"])

print("\n\n=== stop_reason:", stop_reason, "===")

=== Assistant ===


>>> Calling Lambda tool=get_available_vacations_days with {'employee_id': 1}
>>> Lambda returned: {'employee_id': 1, 'available_vacation_days': 7}

=== Assistant (after tool result) ===
Employee 1 currently has **7 vacation days** remaining for this year. If you'd like to reserve any vacation time or have other questions, feel free to ask!

=== stop_reason: end_turn ===


In [30]:
# --- Follow-up prompt: reserve one day off for employee 1 ---
# Reuses collect_stream / invoke_lambda_tool from the cell above.
# The harness now has both get_available_vacations_days and reserve_vacation_time,
# so the model can chain: check balance -> reserve the day.
from datetime import date as _date, timedelta as _timedelta

# Next Monday (weekday 0 = Monday).
_today = _date.today()
_next_monday = _today + _timedelta(days=(7 - _today.weekday()) % 7 or 7)
_target_date = _next_monday.isoformat()

reserve_session_id = str(uuid.uuid4()) + "-hr-reserve-session"  # min 33 chars

reserve_messages = [
    {
        "role": "user",
        "content": [
            {
                "text": (
                    f"Please reserve one vacation day off for employee 1 on "
                    f"{_target_date}."
                )
            }
        ],
    }
]

print("=== Assistant ===")
resp = agentcore_client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=reserve_session_id,
    messages=reserve_messages,
)
assistant_blocks, stop_reason, _ = collect_stream(resp["stream"])

while stop_reason == "tool_use":
    tool_use = next(b["toolUse"] for b in assistant_blocks if "toolUse" in b)
    print(f"\n\n>>> Calling Lambda tool={tool_use['name']} with {tool_use['input']}")
    tool_result = invoke_lambda_tool(tool_use["name"], tool_use["input"])
    print(f">>> Lambda returned: {tool_result}\n")

    followup_messages = [
        {"role": "assistant", "content": assistant_blocks},
        {
            "role": "user",
            "content": [
                {
                    "toolResult": {
                        "toolUseId": tool_use["toolUseId"],
                        "content": [{"text": json.dumps(tool_result)}],
                        "status": "success",
                    }
                }
            ],
        },
    ]

    print("=== Assistant (after tool result) ===")
    resp = agentcore_client.invoke_harness(
        harnessArn=harness_arn,
        runtimeSessionId=reserve_session_id,
        messages=followup_messages,
    )
    assistant_blocks, stop_reason, _ = collect_stream(resp["stream"])

print("\n\n=== stop_reason:", stop_reason, "===")

=== Assistant ===


>>> Calling Lambda tool=reserve_vacation_time with {'employee_id': 1, 'start_date': '2026-08-17'}
>>> Lambda returned: {'employee_id': 1, 'reserved_start_date': '2026-08-17', 'reserved_end_date': '2026-08-17', 'days_reserved': 1, 'remaining_vacation_days': 6, 'status': 'confirmed'}

=== Assistant (after tool result) ===
The vacation day has been successfully reserved! Here's a summary:

- **Employee ID:** 1
- **Date:** August 17, 2026
- **Days Reserved:** 1
- **Remaining Vacation Days:** 6
- **Status:** ✅ Confirmed

Let me know if there's anything else I can help with!

=== stop_reason: end_turn ===


In [48]:
# --- Cleanup: delete every AWS resource this notebook created ---
# Idempotent: safe to re-run. Skips resources that are already gone.
# Local files (employee_database.db, lambda_function.py) are left in place --
# delete them by hand if you want a fully clean checkout.


def _safe(fn, ignore=(), label=""):
    """Call fn() and swallow expected 'not found' style errors."""
    try:
        fn()
        print(f"  deleted {label}")
    except Exception as exc:  # noqa: BLE001
        name = type(exc).__name__
        if any(tok in name for tok in ignore) or any(tok in str(exc) for tok in ignore):
            print(f"  skip    {label} ({name})")
        else:
            print(f"  ERROR   {label}: {name}: {exc}")


print("Deleting AgentCore harness (+ managed memory)...")
try:
    summary = next(
        h
        for h in agentcore_control_client.list_harnesses().get("harnesses", [])
        if h["harnessName"] == harness_name
    )
    hid = summary["harnessId"]

    # Every harness has a DEFAULT endpoint that AgentCore refuses to delete
    # directly -- it goes away as part of DeleteHarness. Only try to delete
    # user-created endpoints here.
    for ep in agentcore_control_client.list_harness_endpoints(harnessId=hid).get(
        "endpoints", []
    ):
        ep_name = ep.get("endpointName") or ep.get("name")
        if not ep_name or ep_name == "DEFAULT":
            print(f"  skip    harness endpoint {ep_name!r} (auto-managed)")
            continue
        # Bind loop vars via default args so the lambda captures this
        # iteration's values, not the last iteration's.
        _safe(
            lambda _hid=hid, _n=ep_name: agentcore_control_client.delete_harness_endpoint(
                harnessId=_hid, endpointName=_n
            ),
            ignore=("NotFound", "ResourceNotFound"),
            label=f"harness endpoint {ep_name}",
        )

    _safe(
        lambda _hid=hid: agentcore_control_client.delete_harness(
            harnessId=_hid, deleteManagedMemory=True
        ),
        ignore=("NotFound", "ResourceNotFound"),
        label=f"harness {harness_name} ({hid})",
    )
except StopIteration:
    print(f"  skip    harness {harness_name} (not found)")

# Sweep for orphan memories -- ONLY unmanaged ones. Memories with
# `managedByResourceArn` set are owned by a harness; either DeleteHarness above
# is still asynchronously reaping them, or you're keeping the owning harness.
# Direct DeleteMemory on a managed record fails with ValidationException.
print("Sweeping any leftover *unmanaged* AgentCore memories...")
for mem in agentcore_control_client.list_memories().get("memories", []):
    mem_id = mem.get("id") or mem.get("memoryId") or ""
    if not mem_id.startswith(f"{harness_name}-"):
        continue
    if mem.get("managedByResourceArn"):
        print(
            f"  skip    memory {mem_id} (managed by "
            f"{mem['managedByResourceArn']}; will be reaped with its harness)"
        )
        continue
    _safe(
        lambda _m=mem_id: agentcore_control_client.delete_memory(memoryId=_m),
        ignore=("NotFound", "ResourceNotFound"),
        label=f"memory {mem_id}",
    )

print("Deleting harness IAM role...")
_safe(
    lambda: iam_client.delete_role_policy(
        RoleName=harness_execution_role_name,
        PolicyName=f"{harness_execution_role_name}-inline",
    ),
    ignore=("NoSuchEntity",),
    label=f"inline policy on {harness_execution_role_name}",
)
_safe(
    lambda: iam_client.delete_role(RoleName=harness_execution_role_name),
    ignore=("NoSuchEntity",),
    label=f"role {harness_execution_role_name}",
)

print("Deleting Lambda function...")
_safe(
    lambda: lambda_client.delete_function(FunctionName=lambda_function_name),
    ignore=("ResourceNotFound",),
    label=f"lambda {lambda_function_name}",
)

print("Deleting Lambda IAM role...")
_safe(
    lambda: iam_client.detach_role_policy(
        RoleName=lambda_function_role,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    ),
    ignore=("NoSuchEntity",),
    label=f"detach AWSLambdaBasicExecutionRole from {lambda_function_role}",
)
_safe(
    lambda: iam_client.delete_role(RoleName=lambda_function_role),
    ignore=("NoSuchEntity",),
    label=f"role {lambda_function_role}",
)

print("Cleanup complete.")

Deleting AgentCore harness (+ managed memory)...
  skip    harness endpoint 'DEFAULT' (auto-managed)
  ERROR   harness hr_assistant_harness (hr_assistant_harness-I5IxgEpmqy): ConflictException: An error occurred (ConflictException) when calling the DeleteHarness operation: Cannot delete agent hr_assistant_harness while it is DELETING. Wait and try again.
Sweeping any leftover *unmanaged* AgentCore memories...
  skip    memory hr_assistant_harness-13Bg7DBdLs (managed by arn:aws:bedrock-agentcore:us-east-1:623271127785:harness/hr_assistant_harness-I5IxgEpmqy; will be reaped with its harness)
Deleting harness IAM role...
  skip    inline policy on AgentCoreHarnessRole_hr_assistant_harness (NoSuchEntityException)
  skip    role AgentCoreHarnessRole_hr_assistant_harness (NoSuchEntityException)
Deleting Lambda function...
  skip    lambda hr-assistant-function-def-us-east-1-623271127785 (ResourceNotFoundException)
Deleting Lambda IAM role...
  skip    detach AWSLambdaBasicExecutionRole from 

In [49]:
# --- Cleanup: delete every AWS resource this notebook created ---
def _safe(fn, ignore=(), label=""):
    try:
        fn()
        print(f"  deleted {label}")
    except Exception as exc:
        name = type(exc).__name__
        if any(tok in name for tok in ignore) or any(tok in str(exc) for tok in ignore):
            print(f"  skip    {label} ({name})")
        else:
            print(f"  ERROR   {label}: {name}: {exc}")


print("Deleting AgentCore harness (+ managed memory)...")
try:
    summary = next(
        h
        for h in agentcore_control_client.list_harnesses().get("harnesses", [])
        if h["harnessName"] == harness_name
    )
    hid = summary["harnessId"]

    for ep in agentcore_control_client.list_harness_endpoints(harnessId=hid).get(
        "endpoints", []
    ):
        ep_name = ep.get("endpointName") or ep.get("name")
        if not ep_name or ep_name == "DEFAULT":
            print(f"  skip    harness endpoint {ep_name!r} (auto-managed)")
            continue
        _safe(
            lambda _hid=hid, _n=ep_name: agentcore_control_client.delete_harness_endpoint(
                harnessId=_hid, endpointName=_n
            ),
            ignore=("NotFound", "ResourceNotFound"),
            label=f"harness endpoint {ep_name}",
        )

    _safe(
        lambda _hid=hid: agentcore_control_client.delete_harness(
            harnessId=_hid, deleteManagedMemory=True
        ),
        ignore=("NotFound", "ResourceNotFound"),
        label=f"harness {harness_name} ({hid})",
    )
except StopIteration:
    print(f"  skip    harness {harness_name} (not found)")

print("Sweeping any leftover *unmanaged* AgentCore memories...")
for mem in agentcore_control_client.list_memories().get("memories", []):
    mem_id = mem.get("id") or mem.get("memoryId") or ""
    if not mem_id.startswith(f"{harness_name}-"):
        continue
    if mem.get("managedByResourceArn"):
        print(
            f"  skip    memory {mem_id} (managed by "
            f"{mem['managedByResourceArn']}; will be reaped with its harness)"
        )
        continue
    _safe(
        lambda _m=mem_id: agentcore_control_client.delete_memory(memoryId=_m),
        ignore=("NotFound", "ResourceNotFound"),
        label=f"memory {mem_id}",
    )

print("Deleting harness IAM role...")
_safe(
    lambda: iam_client.delete_role_policy(
        RoleName=harness_execution_role_name,
        PolicyName=f"{harness_execution_role_name}-inline",
    ),
    ignore=("NoSuchEntity",),
    label=f"inline policy on {harness_execution_role_name}",
)
_safe(
    lambda: iam_client.delete_role(RoleName=harness_execution_role_name),
    ignore=("NoSuchEntity",),
    label=f"role {harness_execution_role_name}",
)

print("Deleting Lambda function...")
_safe(
    lambda: lambda_client.delete_function(FunctionName=lambda_function_name),
    ignore=("ResourceNotFound",),
    label=f"lambda {lambda_function_name}",
)

print("Deleting Lambda IAM role...")
_safe(
    lambda: iam_client.detach_role_policy(
        RoleName=lambda_function_role,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    ),
    ignore=("NoSuchEntity",),
    label=f"detach AWSLambdaBasicExecutionRole from {lambda_function_role}",
)
_safe(
    lambda: iam_client.delete_role(RoleName=lambda_function_role),
    ignore=("NoSuchEntity",),
    label=f"role {lambda_function_role}",
)

print("Cleanup complete.")

Deleting AgentCore harness (+ managed memory)...
  skip    harness hr_assistant_harness (not found)
Sweeping any leftover *unmanaged* AgentCore memories...
  skip    memory hr_assistant_harness-13Bg7DBdLs (managed by arn:aws:bedrock-agentcore:us-east-1:623271127785:harness/hr_assistant_harness-I5IxgEpmqy; will be reaped with its harness)
Deleting harness IAM role...
  skip    inline policy on AgentCoreHarnessRole_hr_assistant_harness (NoSuchEntityException)
  skip    role AgentCoreHarnessRole_hr_assistant_harness (NoSuchEntityException)
Deleting Lambda function...
  skip    lambda hr-assistant-function-def-us-east-1-623271127785 (ResourceNotFoundException)
Deleting Lambda IAM role...
  skip    detach AWSLambdaBasicExecutionRole from hr-assistant-function-def-lambda-role-us-east-1-623271127785 (NoSuchEntityException)
  skip    role hr-assistant-function-def-lambda-role-us-east-1-623271127785 (NoSuchEntityException)
Cleanup complete.


In [50]:
orphan_id = "hr_assistant_harness-13Bg7DBdLs"
try:
    resp = agentcore_control_client.delete_memory(memoryId=orphan_id)
    print("delete_memory response:", resp)
except Exception as exc:
    print(f"ERROR ({type(exc).__name__}): {exc}")

# Verify by re-listing
still_there = [
    m
    for m in agentcore_control_client.list_memories().get("memories", [])
    if (m.get("id") or m.get("memoryId")) == orphan_id
]
print("Still present after delete?", bool(still_there))
if still_there:
    print("  ->", still_there[0])

ERROR (ResourceNotFoundException): An error occurred (ResourceNotFoundException) when calling the DeleteMemory operation: Resource not found during DeleteMemory: Failed to retrieve memory with ID: hr_assistant_harness-13Bg7DBdLs for account: 623271127785
Still present after delete? False
